# Estatísticas do artigo

SVM · `l1_stable` · nocombat. ΔAUC pareado (bootstrap) + FDR BH.
Claim: **`48m_12m`**. Contraste: **Q4 `t1_d21_d32` vs T1 `t1_only`**.
Figs: `6_results.ipynb`. Sem `abs` / `t1_deltas` / `wide`.

| § | Pergunta | Ficheiro |
|---|----------|----------|
| 1 | Demografia sMCI vs pMCI (`48m_12m`) | `stats_demo_48m12` |
| 2 | Idade/sexo confundem imagem? (shape T1) | `stats_confound_48m12` |
| **3** | **Q4 vs T1, 5 famílias, FDR** | **`stats_q4_vs_t1_48m12`** |
| 4 | Mesmo contraste × 4 coortes (células sobrepõem) | `stats_q4_vs_t1_gradient` |
| 5 | Late vs teto shape T1 | `stats_late_vs_shape_48m12` |
| 6 | Clínico vs shape T1 | `stats_clinic_48m12` |
| 7 | Vol Q4 vs leaky | `stats_leaky_48m12` |


In [11]:
from __future__ import annotations

import sys
from dataclasses import dataclass
from pathlib import Path

_MOD = Path.cwd() / "modules"
if str(_MOD) not in sys.path:
    sys.path.insert(0, str(_MOD))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from ablation_analysis import explode_patient_predictions, prepare_ablation_df
from stats_compare import (
    apply_bh_fdr,
    clinical_results_path,
    compare_modalities,
    fusion_results_path,
    image_ablation_path,
    paired_comparison_row,
    print_comparison_summary,
)

COHORT_CLAIM = "48m_12m"
COHORTS_GRADIENT = ("36m_6m", "36m_12m", "48m_6m", "48m_12m")
BASE = Path(f"csvs/cohorts/{COHORT_CLAIM}")
LONG_CSV = BASE / "adnimerged_longitudinal.csv"
TABLE_DIR = Path("artigo") / "tables"

PROTOCOL_BASELINE = "t1_only"
PROTOCOL_LONGITUDINAL = "t1_d21_d32"  # Q4
MODS_COMPARE = ("vol", "shape", "texture", "disp", "firstorder")
CLINIC_MODALITY = "shape"
ALPHA = 0.05

LATE_ALL_T1 = "late__t1_vol__t1_shape__t1_texture__t1_disp__t1_firstorder"
LATE_ALL_Q4 = (
    "late__t1_d21d32_vol__t1_d21d32_shape__t1_d21d32_texture"
    "__t1_d21d32_disp__t1_d21d32_firstorder"
)
LATE_ANCORA = (
    "late__t1_shape__t1_d21d32_vol__t1_d21d32_texture"
    "__t1_d21d32_disp__t1_d21d32_firstorder"
)

FOLDER_REP = {
    "ablation_results_t1_only": "t1_only",
    "ablation_results_d21d32": "t1_d21_d32",
    "ablation_results_leaky_d21d32": "t1_d21_d32",
    "ablation_results_clinic_img_t1_only": "t1_only",
    "ablation_results_late_fusion": "late_fusion",
}


@dataclass(frozen=True)
class StatsCfg:
    task: str = "smci_pmci"
    groups: tuple[str, str] = ("sMCI", "pMCI")
    positive_group: str = "pMCI"
    modality: str = "shape"
    model_key: str = "svm"
    with_combat: bool = False
    selection_mode: str = "l1_stable"
    protocol: str = "t1_only"
    n_perm: int = 5000
    n_bootstrap: int = 5000
    seed: int = 42


STATS_CFG = StatsCfg()


def ler_p(p: float, *, contexto: str = "") -> str:
    tag = "significativo" if p < ALPHA else "não significativo"
    prefix = f"{contexto}: " if contexto else ""
    return f"{prefix}{tag} (p={p:.4f})"


def save_table(df: pd.DataFrame, name: str) -> None:
    TABLE_DIR.mkdir(parents=True, exist_ok=True)
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    print("Salvo:", path)


print("claim:", BASE, "|", PROTOCOL_LONGITUDINAL, "vs", PROTOCOL_BASELINE)
print("mods:", MODS_COMPARE, "| model:", STATS_CFG.model_key, STATS_CFG.selection_mode)
print("perm/boot:", STATS_CFG.n_perm, STATS_CFG.n_bootstrap)


claim: csvs/cohorts/48m_12m | t1_d21_d32 vs t1_only
mods: ('vol', 'shape', 'texture', 'disp', 'firstorder') | model: svm l1_stable
perm/boot: 5000 5000


In [12]:
def load_cohort(long_path: Path, cfg: StatsCfg) -> pd.DataFrame:
    sub = pd.read_csv(long_path)
    sub = sub[sub["slot"].astype(str).isin(("baseline", "t0"))]
    sub = sub.sort_values(["ID_PT", "ID_IMG"]).groupby("ID_PT", as_index=False).first()
    sub = sub[sub["GROUP"].isin(cfg.groups)].copy()
    sub["ID_PT"] = sub["ID_PT"].astype(str)
    sub["y"] = (sub["GROUP"] == cfg.positive_group).astype(int)
    sub["SEX_bin"] = sub["SEX"].map({"M": 0, "F": 1, 0: 0, 1: 1})
    return sub


def permutation_auc_p(
    y, scores, *, n_perm: int, seed: int, bidirectional: bool = False,
) -> tuple[float, float]:
    y = np.asarray(y, dtype=int)
    s = np.asarray(scores, dtype=float)
    auc_obs = float(
        max(roc_auc_score(y, s), roc_auc_score(y, -s)) if bidirectional
        else roc_auc_score(y, s)
    )
    prng = np.random.default_rng(seed)
    ge = 0
    for _ in range(n_perm):
        yp = prng.permutation(y)
        auc_n = (
            max(roc_auc_score(yp, s), roc_auc_score(yp, -s)) if bidirectional
            else roc_auc_score(yp, s)
        )
        ge += int(auc_n >= auc_obs)
    return auc_obs, (ge + 1) / (n_perm + 1)


def bootstrap_auc_diff(y, scores_a, scores_b, *, n_boot: int, seed: int):
    y = np.asarray(y, dtype=int)
    a, b = np.asarray(scores_a, float), np.asarray(scores_b, float)
    obs = float(roc_auc_score(y, a) - roc_auc_score(y, b))
    prng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        idx = prng.integers(0, len(y), size=len(y))
        yb, ab, bb = y[idx], a[idx], b[idx]
        if len(np.unique(yb)) < 2:
            continue
        diffs.append(float(roc_auc_score(yb, ab) - roc_auc_score(yb, bb)))
    diffs = np.asarray(diffs)
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return obs, float(lo), float(hi)


def nested_cv_auc_univariate(X, y, *, seed: int, k: int = 5):
    X = np.asarray(X, dtype=float).reshape(-1, 1)
    y = np.asarray(y, dtype=int)
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=5000, random_state=seed)),
    ])
    cv = StratifiedKFold(k, shuffle=True, random_state=seed)
    scores = cross_val_predict(pipe, X, y, cv=cv, method="predict_proba")[:, 1]
    return float(roc_auc_score(y, scores)), scores


def patient_image_scores(
    path: Path, cfg: StatsCfg, *, expect_representation: str | None = None,
) -> pd.DataFrame:
    raw = prepare_ablation_df(pd.read_csv(path))
    mask = (
        (raw["task"] == cfg.task)
        & (raw["modality"] == cfg.modality)
        & (raw["model_key"] == cfg.model_key)
        & (raw["with_combat"] == cfg.with_combat)
        & (raw["selection_mode"] == cfg.selection_mode)
    )
    if expect_representation is not None and "representation" in raw.columns:
        mask = mask & (raw["representation"].astype(str) == expect_representation)
    sub = raw.loc[mask]
    if sub.empty:
        raise FileNotFoundError(f"sem linhas para {cfg} rep={expect_representation} em {path}")
    pat = explode_patient_predictions(sub)
    return pat.groupby("ID_PT", as_index=False).agg(
        y=("y", "first"), score_img=("score", "mean"),
    )


def patient_scores_from_path(path: Path, cfg: StatsCfg) -> pd.DataFrame:
    expect = None
    names = {p.name for p in path.parents} | {path.parent.name}
    for folder, rep in FOLDER_REP.items():
        if folder in names:
            expect = rep
            break
    return patient_image_scores(
        path, cfg, expect_representation=expect,
    ).rename(columns={"score_img": "score"})


def cfg_for_modality(mod: str, cfg: StatsCfg | None = None, **kw) -> StatsCfg:
    c = cfg or STATS_CFG
    d = {f.name: getattr(c, f.name) for f in StatsCfg.__dataclass_fields__.values()}
    d["modality"] = mod
    d.update(kw)
    return StatsCfg(**d)


def cfg_clinical(cfg: StatsCfg | None = None) -> StatsCfg:
    return cfg_for_modality("clinical", cfg, with_combat=False, selection_mode="none", protocol="clinical")


def compare_q4_vs_t1(
    base: Path,
    cfg: StatsCfg | None = None,
    *,
    comparison: str | None = None,
) -> pd.DataFrame:
    c = cfg or STATS_CFG
    tag = comparison or f"{base.name}_t1_d21_d32_vs_t1_only"
    return compare_modalities(
        MODS_COMPARE,
        path_a=lambda m: image_ablation_path(base, PROTOCOL_LONGITUDINAL, m),
        path_b=lambda m: image_ablation_path(base, PROTOCOL_BASELINE, m),
        cfg_for_mod=lambda m: cfg_for_modality(m, c),
        load_patients=patient_scores_from_path,
        permutation_auc_p=permutation_auc_p,
        n_perm=c.n_perm,
        n_bootstrap=c.n_bootstrap,
        seed=c.seed,
        label_a="q4",
        label_b="t1_only",
        comparison=tag,
        alpha=ALPHA,
    )


cohort = load_cohort(LONG_CSV, STATS_CFG)
g0, g1 = STATS_CFG.groups
print(f"{BASE.name}: n={len(cohort)} | {cohort['GROUP'].value_counts().to_dict()}")


48m_12m: n=120 | {'sMCI': 72, 'pMCI': 48}


## 1. Demografia — `48m_12m`

Idade (Mann-Whitney) e sexo (χ²) entre sMCI e pMCI.


In [13]:
age_a = cohort.loc[cohort["GROUP"] == g0, "AGE"].dropna()
age_b = cohort.loc[cohort["GROUP"] == g1, "AGE"].dropna()
_, p_age_groups = stats.mannwhitneyu(age_a, age_b, alternative="two-sided")
sex_tab = pd.crosstab(
    cohort["GROUP"], cohort["SEX_bin"], rownames=["GROUP"], colnames=["SEX (0=M, 1=F)"],
)
_, p_sex_groups, _, _ = stats.chi2_contingency(sex_tab)
demo = pd.DataFrame([
    {
        "variavel": "idade", "teste": "Mann-Whitney U",
        "media_sMCI": float(age_a.mean()), "media_pMCI": float(age_b.mean()),
        "p": p_age_groups,
    },
    {
        "variavel": "sexo", "teste": "chi2",
        "prop_F_sMCI": float(cohort.loc[cohort["GROUP"] == g0, "SEX_bin"].mean()),
        "prop_F_pMCI": float(cohort.loc[cohort["GROUP"] == g1, "SEX_bin"].mean()),
        "p": p_sex_groups,
    },
])
save_table(demo, "stats_demo_48m12")
display(sex_tab)
display(demo.round(4))
print(ler_p(p_age_groups, contexto="Idade"))
print(ler_p(p_sex_groups, contexto="Sexo"))


Salvo: artigo/tables/stats_demo_48m12.csv


"SEX (0=M, 1=F)",0,1
GROUP,,
pMCI,28,20
sMCI,49,23


,variavel,teste,media_sMCI,media_pMCI,p,prop_F_sMCI,prop_F_pMCI
0,idade,Mann-Whitney U,72.875,73.9792,0.5196,NaN,NaN
1,sexo,chi2,NaN,NaN,0.3714,0.3194,0.4167


Idade: não significativo (p=0.5196)
Sexo: não significativo (p=0.3714)


## 2. Idade/sexo vs imagem (shape T1)

Nested CV univariado + permutação. ΔAUC bootstrap: shape T1 − demografia.


In [14]:
# cfg = STATS_CFG
# pt = cohort[["ID_PT", "y", "AGE", "SEX_bin"]].dropna().copy()
# y = pt["y"].to_numpy()
# auc_age, scores_age = nested_cv_auc_univariate(pt["AGE"].to_numpy(), y, seed=cfg.seed)
# _, p_age = permutation_auc_p(y, scores_age, n_perm=cfg.n_perm, seed=cfg.seed)
# auc_sex, scores_sex = nested_cv_auc_univariate(pt["SEX_bin"].to_numpy(), y, seed=cfg.seed + 1)
# _, p_sex = permutation_auc_p(y, scores_sex, n_perm=cfg.n_perm, seed=cfg.seed + 1)
# print(f"idade AUC={auc_age:.3f}  {ler_p(p_age)}")
# print(f"sexo  AUC={auc_sex:.3f}  {ler_p(p_sex)}")

# pt = pt.assign(score_age=scores_age, score_sex=scores_sex)
# img_path = image_ablation_path(BASE, PROTOCOL_BASELINE, cfg.modality)
# img = patient_image_scores(img_path, cfg, expect_representation="t1_only").merge(
#     pt, on=["ID_PT", "y"], how="inner",
# )
# y_m = img["y"].to_numpy()
# auc_img, p_img = permutation_auc_p(
#     y_m, img["score_img"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 2,
# )
# d_age, lo_age, hi_age = bootstrap_auc_diff(
#     y_m, img["score_img"].to_numpy(), img["score_age"].to_numpy(),
#     n_boot=cfg.n_bootstrap, seed=cfg.seed + 3,
# )
# d_sex, lo_sex, hi_sex = bootstrap_auc_diff(
#     y_m, img["score_img"].to_numpy(), img["score_sex"].to_numpy(),
#     n_boot=cfg.n_bootstrap, seed=cfg.seed + 4,
# )
# confound = pd.DataFrame([
#     {"modelo": "idade", "auc": auc_age, "p_perm": p_age, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "sexo", "auc": auc_sex, "p_perm": p_sex, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "shape T1", "auc": auc_img, "p_perm": p_img, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
#     {"modelo": "shape T1 − idade", "auc": d_age, "p_perm": np.nan, "delta_vs_img": d_age, "ci95_lo": lo_age, "ci95_hi": hi_age},
#     {"modelo": "shape T1 − sexo", "auc": d_sex, "p_perm": np.nan, "delta_vs_img": d_sex, "ci95_lo": lo_sex, "ci95_hi": hi_sex},
# ])
# save_table(confound, "stats_confound_48m12")
# display(confound.round(4))
# print(f"shape T1 AUC={auc_img:.3f} n={len(img)}  {ler_p(p_img)}")
# print(f"Δ vs idade {d_age:.3f} [{lo_age:.3f}, {hi_age:.3f}]")
# print(f"Δ vs sexo  {d_sex:.3f} [{lo_sex:.3f}, {hi_sex:.3f}]")


# §2 — idade / sexo no protocolo do claim (SVM nested 5×10, Optuna 10)
# selection_mode=none: 1 coluna, sem ℓ1. Mesmos knobs que 5_clinic_img.py.

import importlib.util
import json


from ablation_analysis import patient_mean_auc, patient_mean_predictions
from ablation_optuna import tune_pipeline
from ablation_runner import (
    TASKS,
    fold_metrics,
    gridsearch_n_jobs,
    patient_labels_from_long,
    repeat_ids,
    tune_youden_threshold,
)

_spec = importlib.util.spec_from_file_location("clinic_img", Path.cwd() / "5_clinic_img.py")
clinic = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(clinic)

N_REPEATS = 10          # 1 = smoke
OPTUNA_TRIALS = 10
MODEL_KEY = "svm"
cfg = STATS_CFG

MERGE_LONG = BASE / "ablation" / "hippocampus" / "merge_long.csv"
df_long = pd.read_csv(MERGE_LONG)


def nested_cv_one_col(df_long, *, col: str, seed: int, repeat_id: int) -> pd.DataFrame:
    task = TASKS["smci_pmci"]
    clinical = clinic.baseline_clinical_table(df_long)
    if clinical[col].isna().any():
        raise ValueError(f"NaN em {col}")
    pt = patient_labels_from_long(df_long, task)
    y = pt["y"].to_numpy(dtype=int)
    pts = pt["ID_PT"].astype(str).to_numpy()
    outer = StratifiedKFold(5, shuffle=True, random_state=seed)
    rows = []
    for fold, (tr_rel, te_rel) in enumerate(outer.split(np.zeros(len(y)), y), start=1):
        train_pts, test_pts = set(pts[tr_rel]), set(pts[te_rel])
        wide = patient_labels_from_long(df_long, task)[["ID_PT", "GROUP"]].merge(
            clinical, on="ID_PT", how="left",
        )
        wide["y"] = wide["GROUP"].map(task.label_map).astype(int)
        X = wide[[col]].astype(float)
        y_wide = wide["y"].to_numpy(dtype=int)
        tr = np.where(wide["ID_PT"].astype(str).isin(train_pts))[0]
        te = np.where(wide["ID_PT"].astype(str).isin(test_pts))[0]
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y_wide[tr], y_wide[te]
        inner = StratifiedKFold(5, shuffle=True, random_state=seed + fold)
        tune_res = tune_pipeline(
            clinic._clinical_pipeline(MODEL_KEY, seed),
            X_tr, y_tr, inner,
            model_key=MODEL_KEY,
            selection_mode="raw",
            tuner="optuna",
            n_trials=OPTUNA_TRIALS,
            seed=seed + fold,
            n_jobs=gridsearch_n_jobs(MODEL_KEY),
            param_grid=clinic._clinical_param_grid(MODEL_KEY),
        )
        best = tune_res.estimator
        oof = cross_val_predict(best, X_tr, y_tr, cv=inner, method="predict_proba")[:, 1]
        thr = tune_youden_threshold(y_tr, oof)
        scores = best.predict_proba(X_te)[:, 1]
        preds = (scores >= thr).astype(int)
        selected = clinic._clinical_selected_names(
            MODEL_KEY, best.named_steps["clf"], [col],
        )
        rows.append({
            "feature_set": f"demo_{col.lower()}",
            "task": "smci_pmci",
            "modality": col.lower(),
            "model_key": MODEL_KEY,
            "with_combat": False,
            "selection_mode": "none",
            "repeat_id": repeat_id,
            "fold": fold,
            "test_id_pts": json.dumps(wide.iloc[te]["ID_PT"].astype(str).tolist()),
            "test_y_true": json.dumps(y_te.tolist()),
            "test_scores": json.dumps(scores.tolist()),
            "selected_features": json.dumps(selected),
            **fold_metrics(y_te, scores, preds),
        })
    return pd.DataFrame(rows)


def run_demo_col(col: str) -> pd.DataFrame:
    chunks = []
    for rid in repeat_ids(N_REPEATS):
        print(f"{col}  repeat {rid + 1}/{N_REPEATS}", flush=True)
        chunks.append(nested_cv_one_col(
            df_long, col=col,
            seed=cfg.seed + rid * 1000,
            repeat_id=rid,
        ))
    return prepare_ablation_df(pd.concat(chunks, ignore_index=True))


raw_age = run_demo_col("AGE")
raw_sex = run_demo_col("SEX")

pat_age = explode_patient_predictions(raw_age).groupby("ID_PT", as_index=False).agg(
    y=("y", "first"), score_age=("score", "mean"),
)
pat_sex = explode_patient_predictions(raw_sex).groupby("ID_PT", as_index=False).agg(
    y=("y", "first"), score_sex=("score", "mean"),
)
pt = pat_age.merge(pat_sex[["ID_PT", "score_sex"]], on="ID_PT")
y = pt["y"].to_numpy()
auc_age, p_age = permutation_auc_p(y, pt["score_age"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed)
auc_sex, p_sex = permutation_auc_p(y, pt["score_sex"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 1)
print(f"idade SVM  AUC={auc_age:.3f}  {ler_p(p_age)}")
print(f"sexo  SVM  AUC={auc_sex:.3f}  {ler_p(p_sex)}")

img = patient_image_scores(
    image_ablation_path(BASE, PROTOCOL_BASELINE, cfg.modality),
    cfg, expect_representation="t1_only",
).merge(pt, on=["ID_PT", "y"], how="inner")
y_m = img["y"].to_numpy()
auc_img, p_img = permutation_auc_p(y_m, img["score_img"].to_numpy(), n_perm=cfg.n_perm, seed=cfg.seed + 2)
d_age, lo_age, hi_age = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_age"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 3,
)
d_sex, lo_sex, hi_sex = bootstrap_auc_diff(
    y_m, img["score_img"].to_numpy(), img["score_sex"].to_numpy(),
    n_boot=cfg.n_bootstrap, seed=cfg.seed + 4,
)
confound = pd.DataFrame([
    {"modelo": "idade (SVM nested)", "auc": auc_age, "p_perm": p_age, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "sexo (SVM nested)", "auc": auc_sex, "p_perm": p_sex, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "shape T1", "auc": auc_img, "p_perm": p_img, "delta_vs_img": np.nan, "ci95_lo": np.nan, "ci95_hi": np.nan},
    {"modelo": "shape T1 − idade", "auc": d_age, "p_perm": np.nan, "delta_vs_img": d_age, "ci95_lo": lo_age, "ci95_hi": hi_age},
    {"modelo": "shape T1 − sexo", "auc": d_sex, "p_perm": np.nan, "delta_vs_img": d_sex, "ci95_lo": lo_sex, "ci95_hi": hi_sex},
])
save_table(confound, "stats_confound_48m12")
display(confound.round(4))
print(f"shape T1 AUC={auc_img:.3f} n={len(img)}  {ler_p(p_img)}")
print(f"Δ vs idade {d_age:.3f} [{lo_age:.3f}, {hi_age:.3f}]")
print(f"Δ vs sexo  {d_sex:.3f} [{lo_sex:.3f}, {hi_sex:.3f}]")

AGE  repeat 1/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


AGE  repeat 2/10
AGE  repeat 3/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


AGE  repeat 4/10
AGE  repeat 5/10


...

AGE  repeat 10/10


/mnt/study-data/pgirardi/graphs/.venv/lib/python3.12/site-packages/sklearn/svm/_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=2000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


SEX  repeat 1/10
SEX  repeat 2/10
SEX  repeat 3/10
SEX  repeat 4/10
SEX  repeat 5/10
SEX  repeat 6/10
SEX  repeat 7/10
SEX  repeat 8/10
SEX  repeat 9/10
SEX  repeat 10/10
idade SVM  AUC=0.464  não significativo (p=0.7403)
sexo  SVM  AUC=0.385  não significativo (p=0.9820)
Salvo: artigo/tables/stats_confound_48m12.csv


,modelo,auc,p_perm,delta_vs_img,ci95_lo,ci95_hi
0,idade (SVM nested),0.4641,0.7403,NaN,NaN,NaN
1,sexo (SVM nested),0.3854,0.9820,NaN,NaN,NaN
2,shape T1,0.7593,0.0002,NaN,NaN,NaN
3,shape T1 − idade,0.2951,NaN,0.2951,0.1450,0.440
4,shape T1 − sexo,0.3738,NaN,0.3738,0.2325,0.514


shape T1 AUC=0.759 n=120  significativo (p=0.0002)
Δ vs idade 0.295 [0.145, 0.440]
Δ vs sexo  0.374 [0.233, 0.514]


## 3. Claim — Q4 vs T1 em `48m_12m` (teste principal)

Pareado por paciente. H1 one-sided: AUC(Q4) > AUC(T1).
FDR BH nas 5 famílias. Δ descritivo grande em vol/disp; shape empatado (teto).


In [15]:
cfg = STATS_CFG
print("Q4:", image_ablation_path(BASE, PROTOCOL_LONGITUDINAL, "vol").parent.parent.name)
print("T1:", image_ablation_path(BASE, PROTOCOL_BASELINE, "vol").parent.parent.name)
print("mods:", MODS_COMPARE)

cmp_claim = compare_q4_vs_t1(BASE, cfg, comparison=f"{COHORT_CLAIM}_t1_d21_d32_vs_t1_only")
save_table(cmp_claim, "stats_q4_vs_t1_48m12")
print(f"\nClaim | {COHORT_CLAIM} | t1_d21_d32 vs t1_only | {cfg.task} | {cfg.model_key}\n")
display(cmp_claim.round(4))
print("\n── ΔAUC = Q4 − T1; FDR ──")
print_comparison_summary(cmp_claim, label_a="q4", label_b="t1_only")

if not cmp_claim.empty:
    print("\n── Mapa ──")
    for _, r in cmp_claim.iterrows():
        tag = "FDR+" if r["significant_fdr"] else ("raw+" if r["significant_raw"] else "n.s.")
        print(
            f"  {r['modality']}: Q4={r['auc_q4']:.3f}  T1={r['auc_t1_only']:.3f}  "
            f"Δ={r['delta_auc']:+.3f}  [{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]  "
            f"p={r['p_bootstrap_one_sided']:.4f}  q={r['p_fdr_bh']:.4f}  → {tag}"
        )


Q4: ablation_results_d21d32
T1: ablation_results_t1_only
mods: ('vol', 'shape', 'texture', 'disp', 'firstorder')
Salvo: artigo/tables/stats_q4_vs_t1_48m12.csv

Claim | 48m_12m | t1_d21_d32 vs t1_only | smci_pmci | svm



,n_pacientes,auc_q4,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,q4_superior,p_perm_q4,p_perm_t1_only,significant_fdr,significant_raw,modality,comparison,p_fdr_bh
0,120,0.7295,0.6409,0.0885,0.0079,0.1713,0.0164,0.0328,True,0.0002,0.0054,False,True,vol,48m_12m_t1_d21_d32_vs_t1_only,0.0820
1,120,0.7610,0.7593,0.0017,-0.0434,0.0429,0.4711,0.9422,False,0.0002,0.0002,False,False,shape,48m_12m_t1_d21_d32_vs_t1_only,0.5889
2,120,0.5770,0.5286,0.0483,-0.0759,0.1699,0.2274,0.4547,False,0.0802,0.2847,False,False,texture,48m_12m_t1_d21_d32_vs_t1_only,0.5296
3,120,0.6004,0.6160,-0.0156,-0.1117,0.0803,0.6091,0.7818,False,0.0312,0.0170,False,False,disp,48m_12m_t1_d21_d32_vs_t1_only,0.6091
4,120,0.7031,0.6861,0.0171,-0.0532,0.0905,0.3177,0.6355,False,0.0002,0.0004,False,False,firstorder,48m_12m_t1_d21_d32_vs_t1_only,0.5296



── ΔAUC = Q4 − T1; FDR ──
  vol: ΔAUC=0.089 [0.008, 0.171]  p=0.0164  q=0.0820  → raw sig.
  shape: ΔAUC=0.002 [-0.043, 0.043]  p=0.4711  q=0.5889  → sem evidência
  texture: ΔAUC=0.048 [-0.076, 0.170]  p=0.2274  q=0.5296  → sem evidência
  disp: ΔAUC=-0.016 [-0.112, 0.080]  p=0.6091  q=0.6091  → sem evidência
  firstorder: ΔAUC=0.017 [-0.053, 0.091]  p=0.3177  q=0.5296  → sem evidência

── Mapa ──
  vol: Q4=0.729  T1=0.641  Δ=+0.089  [0.008, 0.171]  p=0.0164  q=0.0820  → raw+
  shape: Q4=0.761  T1=0.759  Δ=+0.002  [-0.043, 0.043]  p=0.4711  q=0.5889  → n.s.
  texture: Q4=0.577  T1=0.529  Δ=+0.048  [-0.076, 0.170]  p=0.2274  q=0.5296  → n.s.
  disp: Q4=0.600  T1=0.616  Δ=-0.016  [-0.112, 0.080]  p=0.6091  q=0.6091  → n.s.
  firstorder: Q4=0.703  T1=0.686  Δ=+0.017  [-0.053, 0.091]  p=0.3177  q=0.5296  → n.s.


## 4. Sensibilidade — mesmo contraste × 4 coortes

Células sobrepõem sujeitos. `36m_12m` inverte (pMCI n=33). Não é réplica.
FDR **dentro** de cada coorte (5 famílias).


In [16]:
cfg = STATS_CFG
rows = []
for cname in COHORTS_GRADIENT:
    base = Path(f"csvs/cohorts/{cname}")
    if cname == COHORT_CLAIM and "cmp_claim" in dir() and not cmp_claim.empty:
        cmp = cmp_claim.copy()
    else:
        cmp = compare_q4_vs_t1(base, cfg, comparison=f"{cname}_t1_d21_d32_vs_t1_only")
    if cmp.empty:
        print(f"{cname}: sem dados")
        continue
    cmp = cmp.copy()
    cmp["cohort"] = cname
    rows.append(cmp)
    print(f"\n=== {cname} ===")
    print_comparison_summary(cmp, label_a="q4", label_b="t1_only")

cmp_gradient = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if not cmp_gradient.empty:
    cols = [
        "cohort", "modality", "n_pacientes",
        "auc_q4", "auc_t1_only", "delta_auc",
        "ci95_lo", "ci95_hi", "p_bootstrap_one_sided", "p_fdr_bh",
        "significant_raw", "significant_fdr",
    ]
    save_table(cmp_gradient[cols], "stats_q4_vs_t1_gradient")
    display(cmp_gradient[cols].round(4))
    print("\n── pivot ΔAUC ──")
    display(
        cmp_gradient.pivot(index="modality", columns="cohort", values="delta_auc")
        .reindex(index=list(MODS_COMPARE), columns=list(COHORTS_GRADIENT))
        .round(3)
    )



=== 36m_6m ===
  vol: ΔAUC=0.005 [-0.025, 0.034]  p=0.3803  q=0.6339  → sem evidência
  shape: ΔAUC=-0.003 [-0.032, 0.025]  p=0.5909  q=0.7386  → sem evidência
  texture: ΔAUC=0.070 [0.005, 0.136]  p=0.0178  q=0.0890  → raw sig.
  disp: ΔAUC=-0.029 [-0.077, 0.019]  p=0.8826  q=0.8826  → sem evidência
  firstorder: ΔAUC=0.016 [-0.013, 0.045]  p=0.1468  q=0.3669  → sem evidência

=== 36m_12m ===
  vol: ΔAUC=-0.006 [-0.087, 0.074]  p=0.5521  q=0.7902  → sem evidência
  shape: ΔAUC=-0.027 [-0.093, 0.032]  p=0.7902  q=0.7902  → sem evidência
  texture: ΔAUC=0.016 [-0.074, 0.110]  p=0.3591  q=0.7902  → sem evidência
  disp: ΔAUC=0.178 [0.053, 0.300]  p=0.0042  q=0.0210  → FDR sig.
  firstorder: ΔAUC=-0.025 [-0.119, 0.072]  p=0.6953  q=0.7902  → sem evidência

=== 48m_6m ===
  vol: ΔAUC=0.007 [-0.035, 0.049]  p=0.3639  q=0.8106  → sem evidência
  shape: ΔAUC=-0.010 [-0.033, 0.012]  p=0.8106  q=0.8106  → sem evidência
  texture: ΔAUC=0.047 [-0.022, 0.115]  p=0.0946  q=0.4729  → sem evidência


,cohort,modality,n_pacientes,auc_q4,auc_t1_only,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_fdr_bh,significant_raw,significant_fdr
0,36m_6m,vol,231,0.7703,0.7656,0.0047,-0.0250,0.0336,0.3803,0.6339,False,False
1,36m_6m,shape,231,0.7875,0.7908,-0.0032,-0.0321,0.0253,0.5909,0.7386,False,False
2,36m_6m,texture,231,0.6891,0.6186,0.0705,0.0046,0.1362,0.0178,0.0890,True,False
3,36m_6m,disp,231,0.5737,0.6029,-0.0293,-0.0768,0.0192,0.8826,0.8826,False,False
4,36m_6m,firstorder,231,0.6695,0.6538,0.0157,-0.0134,0.0455,0.1468,0.3669,False,False
5,36m_12m,vol,154,0.6837,0.6900,-0.0063,-0.0870,0.0744,0.5521,0.7902,False,False
6,36m_12m,shape,154,0.7325,0.7596,-0.0270,-0.0930,0.0318,0.7902,0.7902,False,False
7,36m_12m,texture,154,0.5159,0.4996,0.0163,-0.0743,0.1095,0.3591,0.7902,False,False
8,36m_12m,disp,154,0.5167,0.3386,0.1781,0.0528,0.2997,0.0042,0.0210,True,True
9,36m_12m,firstorder,154,0.5733,0.5983,-0.0250,-0.1187,0.0723,0.6953,0.7902,False,False



── pivot ΔAUC ──


cohort,36m_6m,36m_12m,48m_6m,48m_12m
modality,,,,
vol,0.005,-0.006,0.007,0.089
shape,-0.003,-0.027,-0.010,0.002
texture,0.070,0.016,0.047,0.048
disp,-0.029,0.178,-0.018,-0.016
firstorder,0.016,-0.025,-0.000,0.017


## 5. Late vs teto unimodal (shape T1)

Specs: união T1, união Q4, âncora (`shape T1 ∪ resto Q4`). ΔAUC = late − shape T1.


In [17]:
cfg = STATS_CFG
shape_cfg = cfg_for_modality("shape", cfg)
shape_path = image_ablation_path(BASE, PROTOCOL_BASELINE, "shape")
pat_shape = patient_scores_from_path(shape_path, shape_cfg)

late_rows = []
for i, (lab, proto) in enumerate((
    ("late T1", LATE_ALL_T1),
    ("late Q4", LATE_ALL_Q4),
    ("late ancora", LATE_ANCORA),
)):
    path = image_ablation_path(BASE, proto, proto)
    if not path.is_file():
        print("MISSING", path)
        continue
    late_cfg = cfg_for_modality(proto, cfg, protocol=proto)
    pat_late = patient_scores_from_path(path, late_cfg)
    paired = pat_late.merge(pat_shape, on=["ID_PT", "y"], suffixes=("_late", "_shape"))
    if paired.empty:
        print("sem pares", lab)
        continue
    row = paired_comparison_row(
        paired,
        score_a="score_late",
        score_b="score_shape",
        label_a="late",
        label_b="shape_t1",
        n_boot=cfg.n_bootstrap,
        seed=cfg.seed + 400 + i,
        permutation_auc_p=permutation_auc_p,
        n_perm=cfg.n_perm,
        alpha=ALPHA,
    )
    row.update({"spec": lab, "protocol": proto, "modality": lab})
    late_rows.append(row)

cmp_late = pd.DataFrame(late_rows)
if not cmp_late.empty:
    cmp_late["p_fdr_bh"] = apply_bh_fdr(cmp_late["p_bootstrap_one_sided"].to_numpy())
    cmp_late["significant_fdr"] = (cmp_late["p_fdr_bh"] < ALPHA) & (cmp_late["ci95_lo"] > 0)
    save_table(cmp_late, "stats_late_vs_shape_48m12")
    display(cmp_late.round(4))
    print("\n── ΔAUC = late − shape T1 ──")
    print_comparison_summary(cmp_late, label_a="late", label_b="shape_t1")


Salvo: artigo/tables/stats_late_vs_shape_48m12.csv


,n_pacientes,auc_late,auc_shape_t1,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,late_superior,p_perm_late,p_perm_shape_t1,significant_fdr,significant_raw,spec,protocol,modality,p_fdr_bh
0,120,0.7405,0.7593,-0.0188,-0.0798,0.0424,0.7239,0.5523,False,0.0002,0.0002,False,False,late T1,late__t1_vol__t1_shape__t1_texture__t1_disp__t...,late T1,0.7239
1,120,0.7648,0.7593,0.0055,-0.0646,0.0732,0.4439,0.8878,False,0.0002,0.0002,False,False,late Q4,late__t1_d21d32_vol__t1_d21d32_shape__t1_d21d3...,late Q4,0.6659
2,120,0.7668,0.7593,0.0075,-0.0547,0.0693,0.3951,0.7902,False,0.0002,0.0002,False,False,late ancora,late__t1_shape__t1_d21d32_vol__t1_d21d32_textu...,late ancora,0.6659



── ΔAUC = late − shape T1 ──
  late T1: ΔAUC=-0.019 [-0.080, 0.042]  p=0.7239  q=0.7239  → sem evidência
  late Q4: ΔAUC=0.005 [-0.065, 0.073]  p=0.4439  q=0.6659  → sem evidência
  late ancora: ΔAUC=0.008 [-0.055, 0.069]  p=0.3951  q=0.6659  → sem evidência


## 6. Clínico vs shape T1 (`48m_12m`)

Imagem = shape `t1_only` (não abs/wide). Fusão = clinic + shape T1.


In [18]:
cfg = STATS_CFG
mod = CLINIC_MODALITY
img_cfg = cfg_for_modality(mod, cfg)
clin_cfg = cfg_clinical(cfg)
fusion_path = fusion_results_path(
    BASE, mod, selection_mode=cfg.selection_mode, with_combat=cfg.with_combat,
    representation="t1_only",
)
clin_path = clinical_results_path(BASE)
img_path = image_ablation_path(BASE, PROTOCOL_BASELINE, mod)

clinical_rows = []
pairs = [
    (img_path, clin_path, img_cfg, clin_cfg, "score_img", "score_clin", "img", "clin", "shape_t1_vs_clinical"),
    (fusion_path, clin_path, img_cfg, clin_cfg, "score_fusion", "score_clin", "fusion", "clin", "fusion_vs_clinical"),
    (fusion_path, img_path, img_cfg, img_cfg, "score_fusion", "score_img", "fusion", "img", "fusion_vs_shape_t1"),
]
for i, (pa, pb, ca, cb, sa, sb, la, lb, tag) in enumerate(pairs):
    if not pa.exists() or not pb.exists():
        print("MISSING", pa if not pa.exists() else pb)
        continue
    paired = patient_scores_from_path(pa, ca).merge(
        patient_scores_from_path(pb, cb), on=["ID_PT", "y"], suffixes=(f"_{la}", f"_{lb}"),
    )
    # suffixes only apply when column names collide; scores both named "score"
    if f"score_{la}" not in paired.columns:
        paired = patient_scores_from_path(pa, ca).rename(columns={"score": sa}).merge(
            patient_scores_from_path(pb, cb).rename(columns={"score": sb}),
            on=["ID_PT", "y"],
        )
    row = paired_comparison_row(
        paired, score_a=sa, score_b=sb, label_a=la, label_b=lb,
        n_boot=cfg.n_bootstrap, seed=cfg.seed + 200 + i,
        permutation_auc_p=permutation_auc_p, n_perm=cfg.n_perm, alpha=ALPHA,
    )
    row.update({"modality": mod, "comparison": tag})
    clinical_rows.append(row)

cmp_clinical = pd.DataFrame(clinical_rows)
if not cmp_clinical.empty:
    cmp_clinical["p_fdr_bh"] = apply_bh_fdr(cmp_clinical["p_bootstrap_one_sided"].to_numpy())
    cmp_clinical["significant_fdr"] = (
        (cmp_clinical["p_fdr_bh"] < ALPHA) & (cmp_clinical["ci95_lo"] > 0)
    )
    save_table(cmp_clinical, "stats_clinic_48m12")
    display(cmp_clinical.round(4))
    for _, r in cmp_clinical.iterrows():
        print(
            f"  {r['comparison']}: Δ={r['delta_auc']:.3f} "
            f"[{r['ci95_lo']:.3f}, {r['ci95_hi']:.3f}]  p={r['p_bootstrap_one_sided']:.4f}"
        )


Salvo: artigo/tables/stats_clinic_48m12.csv


,n_pacientes,auc_img,auc_clin,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,img_superior,p_perm_img,p_perm_clin,significant_fdr,significant_raw,modality,comparison,auc_fusion,fusion_superior,p_perm_fusion,p_fdr_bh
0,120,0.7593,0.8119,-0.0527,-0.1581,0.0545,0.8430,0.3139,False,0.0002,0.0002,False,False,shape,shape_t1_vs_clinical,NaN,NaN,NaN,0.8430
1,120,NaN,0.8119,0.0339,-0.0171,0.0858,0.0968,0.1936,NaN,NaN,0.0002,False,False,shape,fusion_vs_clinical,0.8458,False,0.0002,0.1452
2,120,0.7593,NaN,0.0865,0.0216,0.1565,0.0058,0.0116,NaN,0.0002,NaN,True,True,shape,fusion_vs_shape_t1,0.8458,True,0.0002,0.0174


  shape_t1_vs_clinical: Δ=-0.053 [-0.158, 0.055]  p=0.8430
  fusion_vs_clinical: Δ=0.034 [-0.017, 0.086]  p=0.0968
  fusion_vs_shape_t1: Δ=0.087 [0.022, 0.157]  p=0.0058


## 7. Leaky — vol Q4 vs vol Q4 com stats globais

`t1_d21_d32` vs `t1_d21_d32_global` (só vol no disco). ΔAUC = Q4 correcto − leaky.


In [19]:
cfg = STATS_CFG
cmp_leaky = compare_modalities(
    ("vol",),
    path_a=lambda m: image_ablation_path(BASE, PROTOCOL_LONGITUDINAL, m),
    path_b=lambda m: image_ablation_path(BASE, "t1_d21_d32_global", m),
    cfg_for_mod=lambda m: cfg_for_modality(m, cfg),
    load_patients=patient_scores_from_path,
    permutation_auc_p=permutation_auc_p,
    n_perm=cfg.n_perm,
    n_bootstrap=cfg.n_bootstrap,
    seed=cfg.seed + 100,
    label_a="q4",
    label_b="leaky",
    comparison="q4_vs_leaky_vol",
    alpha=ALPHA,
)
save_table(cmp_leaky, "stats_leaky_48m12")
display(cmp_leaky.round(4))
print("── ΔAUC = Q4 − leaky (negativo → leaky maior) ──")
print_comparison_summary(cmp_leaky, label_a="q4", label_b="leaky")


Salvo: artigo/tables/stats_leaky_48m12.csv


,n_pacientes,auc_q4,auc_leaky,delta_auc,ci95_lo,ci95_hi,p_bootstrap_one_sided,p_bootstrap_two_sided,q4_superior,p_perm_q4,p_perm_leaky,significant_fdr,significant_raw,modality,comparison,p_fdr_bh
0,120,0.7295,0.7344,-0.0049,-0.0238,0.0154,0.7071,0.5859,False,0.0002,0.0002,False,False,vol,q4_vs_leaky_vol,0.7071


── ΔAUC = Q4 − leaky (negativo → leaky maior) ──
  vol: ΔAUC=-0.005 [-0.024, 0.015]  p=0.7071  q=0.7071  → sem evidência
